# TensorFlow 2 quickstart for beginners

This short introduction uses [Keras](https://www.tensorflow.org/guide/keras) to:

1. Load a prebuilt dataset.
2. Build a neural network machine learning model that classifies images.
3. Train this neural network.
4. Evaluate the accuracy of the model.
5. Train with validation and early stopping, plot the training curves, and save the model.

## How to run this notebook

Run it in Jupyter, JupyterLab or VS Code with a Python 3.9+ kernel. Install TensorFlow (this also installs Keras 3) and Matplotlib (for the plots) first:

```bash
pip install --upgrade pip
pip install tensorflow matplotlib
```

To run everything, choose **Run All**. To run the code cells one at a time, select a cell and press `Shift+Enter`.

## 1. Set up TensorFlow

Import TensorFlow into your program to get started:

In [ ]:
# 1. Set up TensorFlow
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("Keras version:", tf.keras.__version__)

## 2. Load a dataset

Load and prepare the [MNIST](http://yann.lecun.com/exdb/mnist/) dataset of handwritten digits. The pixel values of the images range from 0 through 255. Scale these values to a range of 0 to 1 by dividing the values by `255.0`. This also converts the sample data from integers to floating-point numbers:

In [ ]:
# 2. Load a dataset
mnist = tf.keras.datasets.mnist

(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0

## 3. Build a machine learning model

Build a `tf.keras.Sequential` model. In current Keras the input shape is declared with an explicit `tf.keras.Input` layer instead of the old `input_shape` argument on the first layer:

In [ ]:
# 3. Build a machine learning model
model = tf.keras.Sequential([
    tf.keras.Input(shape=(28, 28)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(10),
])

model.summary()

## 4. Get predictions from the untrained model

[`Sequential`](https://www.tensorflow.org/guide/keras/sequential_model) is useful for stacking layers where each layer has one input [tensor](https://www.tensorflow.org/guide/tensor) and one output tensor. Layers are functions with a known mathematical structure that can be reused and have trainable variables. Most TensorFlow models are composed of layers. This model uses the [`Flatten`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Flatten), [`Dense`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense), and [`Dropout`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dropout) layers.

For each example, the model returns a vector of [logits](https://developers.google.com/machine-learning/glossary#logits) or [log-odds](https://developers.google.com/machine-learning/glossary#log-odds) scores, one for each class.

In [ ]:
# 4. Get predictions from the untrained model
predictions = model(x_train[:1]).numpy()
predictions

## 5. Convert logits to probabilities

The `tf.nn.softmax` function converts these logits to *probabilities* for each class:

In [ ]:
# 5. Convert logits to probabilities
tf.nn.softmax(predictions).numpy()

### Note: why the model outputs logits

It is possible to bake the `tf.nn.softmax` function into the activation function for the last layer of the network. While this can make the model output more directly interpretable, this approach is discouraged as it's impossible to provide an exact and numerically stable loss calculation for all models when using a softmax output.

## 6. Define the loss function

Define a loss function for training using `tf.keras.losses.SparseCategoricalCrossentropy`. It takes a vector of ground truth values and a vector of logits and returns a scalar loss for each example. This loss is equal to the negative log probability of the true class: the loss is zero if the model is sure of the correct class.

This untrained model gives probabilities close to random (1/10 for each class), so the initial loss should be close to `-tf.math.log(1/10) ~= 2.3`.

In [ ]:
# 6. Define the loss function
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

loss_fn(y_train[:1], predictions).numpy()

## 7. Compile the model

Before you start training, configure and compile the model using Keras `Model.compile`. Set the [`optimizer`](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers) class to `adam`, set the `loss` to the `loss_fn` function you defined earlier, and specify a metric to be evaluated for the model by setting the `metrics` parameter to `accuracy`.

In [ ]:
# 7. Compile the model
model.compile(
    optimizer="adam",
    loss=loss_fn,
    metrics=["accuracy"],
)

## 8. Train the model

Use the `Model.fit` method to adjust your model parameters and minimize the loss:

In [ ]:
# 8. Train the model
model.fit(x_train, y_train, epochs=5)

## 9. Evaluate the model

The `Model.evaluate` method checks the model's performance, usually on a [validation set](https://developers.google.com/machine-learning/glossary#validation-set) or [test set](https://developers.google.com/machine-learning/glossary#test-set).

In [ ]:
# 9. Evaluate the model
model.evaluate(x_test, y_test, verbose=2)

### Result

The image classifier is now trained to ~98% accuracy on this dataset.

## 10. Return probabilities from the model

If you want your model to return a probability, you can wrap the trained model, and attach the softmax to it:

In [ ]:
# 10. Return probabilities from the model
probability_model = tf.keras.Sequential([
    model,
    tf.keras.layers.Softmax(),
])

## 11. Predict on test samples

Each row below holds the probability of every digit (0-9) for one of the first five test images:

In [ ]:
# 11. Predict on test samples
probability_model(x_test[:5])

## 12. Example: train a model with validation and early stopping

Steps 3 to 8 trained one model for a fixed number of epochs. In practice you also want to know **when to stop**. This example puts the model into a function so you can build a fresh one on demand, then trains it with:

* `validation_split=0.1` - keeps 10% of the training data aside and reports `val_loss` / `val_accuracy` after every epoch, so you can see overfitting.
* `EarlyStopping` - stops training when `val_loss` has not improved for 2 epochs and restores the best weights.
* `batch_size=32` - how many images are used for each weight update.

In [ ]:
# 12. Example: train a model with validation and early stopping
def build_model():
    """Builds and compiles a fresh, untrained MNIST classifier."""
    new_model = tf.keras.Sequential([
        tf.keras.Input(shape=(28, 28)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(10),
    ])
    new_model.compile(
        optimizer="adam",
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=["accuracy"],
    )
    return new_model


early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True,
)

trained_model = build_model()
history = trained_model.fit(
    x_train,
    y_train,
    validation_split=0.1,
    epochs=10,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=2,
)

## 13. Plot the training curves

`Model.fit` returns a `History` object. Plot the loss and accuracy per epoch: if the validation curve stops improving (or gets worse) while the training curve keeps improving, the model is starting to overfit.

In [ ]:
# 13. Plot the training curves
import matplotlib.pyplot as plt

fig, (ax_loss, ax_accuracy) = plt.subplots(1, 2, figsize=(10, 4))

ax_loss.plot(history.history["loss"], label="train")
ax_loss.plot(history.history["val_loss"], label="validation")
ax_loss.set_title("Loss")
ax_loss.set_xlabel("Epoch")
ax_loss.legend()

ax_accuracy.plot(history.history["accuracy"], label="train")
ax_accuracy.plot(history.history["val_accuracy"], label="validation")
ax_accuracy.set_title("Accuracy")
ax_accuracy.set_xlabel("Epoch")
ax_accuracy.legend()

plt.tight_layout()
plt.show()

## 14. Test the trained model on one image

Evaluate on the held-out test set (data the model never saw during training), then predict one digit and compare it with the real label:

In [ ]:
# 14. Test the trained model on one image
test_loss, test_accuracy = trained_model.evaluate(x_test, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f}, test accuracy: {test_accuracy:.4f}")

index = 0
probabilities = tf.nn.softmax(trained_model(x_test[index:index + 1])).numpy()[0]
predicted_digit = int(probabilities.argmax())

plt.imshow(x_test[index], cmap="gray")
plt.title(f"Predicted: {predicted_digit} ({probabilities[predicted_digit]:.1%}), actual: {y_test[index]}")
plt.axis("off")
plt.show()

## 15. Save and reload the trained model

Save the model in the native Keras format (`.keras`: architecture, weights and optimizer state in one file) so you can use it later without training again. The file is written to the notebook's working directory.

In [ ]:
# 15. Save and reload the trained model
trained_model.save("mnist_model.keras")

reloaded_model = tf.keras.models.load_model("mnist_model.keras")
reloaded_loss, reloaded_accuracy = reloaded_model.evaluate(x_test, y_test, verbose=0)
print(f"Reloaded model - test loss: {reloaded_loss:.4f}, test accuracy: {reloaded_accuracy:.4f}")

## Conclusion

You have trained a machine learning model on a prebuilt dataset using the [Keras](https://www.tensorflow.org/guide/keras) API.

For more, see the TensorFlow [tutorials](https://www.tensorflow.org/tutorials) and the Keras [guides](https://www.tensorflow.org/guide/keras).

## License and attribution

Author: Rohtash Lakra

This notebook is adapted from the TensorFlow 2 beginner quickstart, Copyright 2019 The TensorFlow Authors, and the original work remains under the Apache License, Version 2.0. The changes in this notebook are: updated for Keras 3 (explicit `tf.keras.Input`, `tf.keras.Sequential`, `model.summary()`), restructured into numbered steps, and reworded for local Jupyter use.

Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at <https://www.apache.org/licenses/LICENSE-2.0>.

Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.